In [49]:
from dotenv import load_dotenv
import os
import json

In [50]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText #Текст не вставляю -> не требуется
from email.mime.base import MIMEBase
from email import encoders
from email.utils import formatdate
import imaplib
import smtplib
import email

In [51]:
load_dotenv()

EMAIL_USER = os.getenv("EMAIL_USER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
FORWARD_TO = os.getenv("FORWARD_TO")
ATTACHMENT_PATH = os.getenv("ATTACHMENT_PATH")

In [52]:
YANDEX_SMTP = "smtp.yandex.ru"
YANDEX_IMAP = "imap.yandex.ru"

In [53]:
print(EMAIL_PASSWORD)

efydjkepgbcpnrtr


In [54]:
import mimetypes

def send_email(to_email):
    msg = MIMEMultipart()
    msg["From"] = EMAIL_USER
    msg["To"] = to_email
    msg["Subject"] = "Почта с вложенным файлом"
    msg["Date"] = formatdate(localtime=True)
    msg.attach(MIMEText("Документ", "plain", "utf-8"))
    if not ATTACHMENT_PATH or not os.path.exists(ATTACHMENT_PATH):
        print("Вложение не найдено")
        return
    
    filename = os.path.basename(ATTACHMENT_PATH)
    print(filename)
    ctype, encoding = mimetypes.guess_type(ATTACHMENT_PATH)
    if ctype is None or encoding is not None:
        print("Вложения не правильного формата")
        return
    maintype, subtype = ctype.split('/', 1)
    with open(ATTACHMENT_PATH, "rb") as attachment:
        part = MIMEBase(maintype, subtype)
        part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header('Content-Disposition', f'attachment; filename="{filename}"')
        msg.attach(part)
        
    with smtplib.SMTP_SSL(YANDEX_SMTP, 465) as server:
        server.login(EMAIL_USER, EMAIL_PASSWORD)
        server.sendmail(EMAIL_USER, to_email, msg.as_bytes())
        print("Письмо отправлено")

In [55]:
def send_json():
    with open("emails.json", encoding="utf-8") as file:
        emails = json.load(file)
    for email in emails:
        send_email(email["email"])

In [56]:
send_json()

AVA.docx
Письмо отправлено
AVA.docx
Письмо отправлено


In [67]:
def forward_emails():
    mail = imaplib.IMAP4_SSL(YANDEX_IMAP)
    mail.login(EMAIL_USER, EMAIL_PASSWORD)
    mail.select("inbox")
    status, messages = mail.search(None, "ALL")
    if status != "OK":
        print("Не удалось получить писема")
        return
    # print(messages)

    email_id = messages[0].split()
    
    for id in email_id:
        status, data = mail.fetch(id, "(RFC822)")
        
        if status != "OK":
            continue
        raw_email = data[0][1]
        msg = email.message_from_bytes(raw_email)

        has_attachment = False
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_disposition() == "attachment":
                    has_attachment = True
                    break
        
        if has_attachment:
            print(EMAIL_USER)
            print(FORWARD_TO)
            subject = msg.get("Subject", "(без темы)")
            msg_forward = MIMEMultipart()
            msg_forward["From"] = EMAIL_USER
            msg_forward["To"] = FORWARD_TO
            msg_forward["Subject"] = subject
            msg_forward["Date"] = formatdate(localtime=True)
            msg_forward.attach(MIMEText("Пересланное", "plain", "utf-8"))


            # orig_part = MIMEBase("message", "rfc822")
            # orig_part.set_payload(raw_email)
            # encoders.encode_base64(orig_part)
            # orig_part.add_header("Content-Disposition", "attachment; filename=\"original_message.eml\"")
            # msg_forward.attach(orig_part)

            body_parts = []
            attachments = []

            if msg.is_multipart():
                for part in msg.walk():
                    content_type = part.get_content_type()
                    content_disposition = str(part.get("Content-Disposition"))
                    if "attachment" in content_disposition:
                        attachments.append(part)
                    elif content_type == "text/plain":
                        body_parts.append(part.get_payload(decode=True).decode(part.get_content_charset() or 'utf-8', errors='replace'))
            else:
                payload = msg.get_payload(decode=True)
                if payload:
                    charset = msg.get_content_charset() or 'utf-8'
                    body_parts.append(payload.decode(charset, errors='replace'))
            
            full_body = ""
            full_body += "".join(body_parts)
            msg_forward.attach(MIMEText(full_body, "plain", "utf-8"))

            for part in attachments:
                attachment_part = MIMEBase(part.get_content_type().split('/')[0], part.get_content_type().split('/')[1])
                attachment_part.set_payload(part.get_payload())
                for header, value in part.items():
                    attachment_part[header] = value
                msg_forward.attach(attachment_part)

            with smtplib.SMTP_SSL(YANDEX_SMTP, 465) as server:
                server.login(EMAIL_USER, EMAIL_PASSWORD)
                server.sendmail(EMAIL_USER, FORWARD_TO, msg_forward.as_bytes())
                print("Письмо отправлено")
    mail.close()
    mail.logout()

In [68]:
forward_emails()

egorshakiryanov@yandex.ru
sakiranovegor@mail.ru
Письмо отправлено
